# MINI SHAZAM

## BIBLIOTEKE

In [ ]:
import os
from pathlib import Path
import numpy as np
import random 
from tqdm import tqdm
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F
import torch
from tqdm import tqdm
import torch.optim as optim
from pytorch_metric_learning import losses, miners
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights
import os
import random
import torch
import numpy as np
from torch.utils.data import Dataset

## DIREKTORIJUMI

In [ ]:
ROOT_DIR = str(Path('..').parent.resolve())
SPECT_DIR = "spectrograms-20260626T174438Z-3-001/spectrograms"
NOISE_DIR = "noiseMeowSpectrograms/noiseMeowSpectrograms"

## OGRANICENJA I DEVICES

In [ ]:
torch.cuda.set_per_process_memory_fraction(0.18)  # ~4 GB out of 24 GB

## PREPROCESSING

In [ ]:
def add_background_noise(
    clean_spec: torch.Tensor,
    noise_file_path: str,
    noise_reduction_db: float = 15.0,
) -> torch.Tensor:
    """
    Overlays a background noise spectrogram onto a clean spectrogram.

    clean_spec: Tensor of shape [1, M, T] or [M, T]
    """

    noise_spec = torch.from_numpy(np.load(noise_file_path)).float()

    squeeze_channel = False
    if clean_spec.ndim == 3:
        clean_spec = clean_spec.squeeze(0)
        squeeze_channel = True

    if noise_spec.ndim == 3:
        noise_spec = noise_spec.squeeze(0)

    target_time = clean_spec.shape[1]

    # Loop if necessary
    if noise_spec.shape[1] < target_time:
        repeats = (target_time + noise_spec.shape[1] - 1) // noise_spec.shape[1]
        noise_spec = noise_spec.repeat(1, repeats)

    # Trim
    noise_spec = noise_spec[:, :target_time]

    # Lower noise level
    noise_spec = noise_spec - noise_reduction_db

    # Mix
    noisy_spec = clean_spec + noise_spec

    if squeeze_channel:
        noisy_spec = noisy_spec.unsqueeze(0)

    return noisy_spec

## DATASET

In [ ]:


class SimpleAudioDataset(Dataset):
    def __init__(self, data_paths, labels, noise_paths, type_of_dataset):
        self.data_paths = data_paths
        self.labels = labels
        self.noise_paths = noise_paths
        self.type_of_dataset = type_of_dataset

    def __len__(self):
        return len(self.data_paths)

    def __getitem__(self, index):
        file_path = self.data_paths[index]
        label = self.labels[index]

        # Load the numpy array and convert to a PyTorch float tensor
        db_mel_spec = torch.from_numpy(np.load(file_path)).float()

        # --- FIX: Add explicit channel dimension [Height, Width] -> [1, Height, Width] ---
        if db_mel_spec.ndim == 2:
            db_mel_spec = db_mel_spec.unsqueeze(0)

        if self.type_of_dataset == "train":
            # 50% chance of adding background noise
            if random.random() < 0.5:
                db_mel_spec = add_background_noise(
                    db_mel_spec,
                    random.choice(self.noise_paths),
                    random.randint(-10, 5)  # attenuation in dB
                )
        elif self.type_of_dataset == "test":
            # Test gets noise every time to simulate real-world Shazam usage
            db_mel_spec = add_background_noise(
                    db_mel_spec,
                    random.choice(self.noise_paths),
                    random.randint(-10, 5)  
                )
        elif self.type_of_dataset == "original":
            pass
        else:
            print("Error: Unknown dataset type, returning original spectrogram.")
            
        return db_mel_spec, label

In [ ]:
filenames = []
labels = []

test_filenames = []
test_labels = []

for song in sorted(os.listdir(SPECT_DIR)):
    song_dir = os.path.join(SPECT_DIR, song)

    if not os.path.isdir(song_dir):
        continue

    song_listica = sorted(os.listdir(song_dir))

    if len(song_listica) ==0:
        continue
    
    for filename in song_listica:
        filenames.append(os.path.join(song_dir, filename))
        labels.append(int(song))
    
    test_filenames.append(os.path.join(song_dir, random.choice(song_listica)))
    test_labels.append(int(song))
    


print(len(filenames))
print(len(labels))

print(len(test_filenames))
print(len(test_labels))

print(f"Training on {len(labels)} total files")

40368
40368
448
448
Training on 40368 total files


In [9]:
noise_names =os.listdir(NOISE_DIR)
noise_paths =  [os.path.join(NOISE_DIR, noise_name) for noise_name in noise_names]

In [10]:
print(noise_paths)

['noiseMeowSpectrograms/noiseMeowSpectrograms/freesound_community-noise-32940.npy', 'noiseMeowSpectrograms/noiseMeowSpectrograms/freesound_community-noise-meow32940.npy', 'noiseMeowSpectrograms/noiseMeowSpectrograms/freesound_community-015939_street-cafe-very-busy-motorcycle-stopping-by-56168.npy']


In [11]:
train_dataset = SimpleAudioDataset(data_paths=filenames, labels=labels, noise_paths = noise_paths, type_of_dataset="train")

# 2. Create the DataLoader factory line
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=64,       # Number of audio files to process simultaneously
    shuffle=True,        # Mixes up the order every epoch so the model learns features, not order
    drop_last=True       # Trims the last batch if it doesn't perfectly divide by 64
)

## MODEL

In [ ]:


class AudioResNet(nn.Module):
    def __init__(self, embedding_size=128):
        super().__init__()
        
        # 1. Load the pre-trained ResNet-18 (Transfer Learning)
        # Using pre-trained weights gives the model a massive head start in recognizing audio textures
        self.backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        
        # 2. MODIFY INPUT: Change the first layer to accept 1-Channel audio instead of 3-Channel RGB
        # We keep the exact same kernel size and stride as the original model
        self.backbone.conv1 = nn.Conv2d(
            in_channels=1, 
            out_channels=64, 
            kernel_size=(7, 7), 
            stride=(2, 2), 
            padding=(3, 3), 
            bias=False
        )
        
        # 3. MODIFY OUTPUT: Replace the final Fully Connected (fc) layer
        # Standard ResNet outputs 1000 categories. We need it to output our 128-number embedding.
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(num_features, embedding_size)

    def forward(self, x):
        # Pass the 2D spectrogram tensor through the ResNet backbone
        embeddings = self.backbone(x)
        
        # CRITICAL FOR TRIPLET LOSS: L2 Normalize the embeddings
        # This forces all audio clips to sit on the surface of a perfect mathematical sphere,
        # making the distance calculations highly accurate.
        normalized_embeddings = F.normalize(embeddings, p=2, dim=1)
        
        return normalized_embeddings

In [13]:
train_model = AudioResNet(embedding_size=128)

dummy_spectrograms = torch.randn(64, 1, 128, 216)
    
# Run the fake data through the model
output = train_model(dummy_spectrograms)

print("--- Model Architecture Test ---")
print(f"Input Spectrogram Shape:  {dummy_spectrograms.shape}")
print(f"Output Embedding Shape:   {output.shape}")
print("Test Passed: The model successfully output 128 normalized coordinates per audio clip!")

--- Model Architecture Test ---
Input Spectrogram Shape:  torch.Size([64, 1, 128, 216])
Output Embedding Shape:   torch.Size([64, 128])
Test Passed: The model successfully output 128 normalized coordinates per audio clip!


## LOSS AND OPTIMIZER

In [ ]:


# The Miner: Set to "hard" so it actively looks for the most difficult negatives to tell apart
miner = miners.TripletMarginMiner(margin=1.0, type_of_triplets="hard")

# The Loss Function
loss_func = losses.TripletMarginLoss(margin=1.0)

In [ ]:


# Move the model to your GPU if you have one
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_model = train_model.to(device)

# 2. Initialize the Adam Optimizer
# lr=1e-4 is the safe, standard learning rate for fine-tuning pre-trained models
optimizer = optim.Adam(train_model.parameters(), lr=1e-4)

## TRAINING

In [16]:
MODEL_DIR = os.path.join(ROOT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:


# num_epochs = 20

# for epoch in range(num_epochs):
#     print(f"\n=== EPOCH {epoch+1}/{num_epochs} ===")
    
#     # BEST PRACTICE: Always put the model in train mode INSIDE the loop
#     train_model.train()
    
#     # We put the tqdm progress bar here!
#     loop = tqdm(train_dataloader)
    
#     for batch in loop:
#         # 1. Get the normal batch of data
#         spectrograms, labels_batch = batch
#         spectrograms, labels_batch = spectrograms.to(device), labels_batch.to(device)
        
#         optimizer.zero_grad()

#         # 2. Pass the entire batch through the CNN to get embeddings
#         embeddings = train_model(spectrograms)

#         # 3. NINE-LINE MAGIC: The Miner finds the hardest triplets in this batch!
#         indices_tuple = miner(embeddings, labels_batch)

#         # 4. Calculate the loss using ONLY the hard triplets found by the miner
#         loss = loss_func(embeddings, labels_batch, indices_tuple)

#         # 5. Backpropagate and update
#         loss.backward()
#         optimizer.step()

#         # Update the progress bar text dynamically so you can watch it drop
#         loop.set_postfix(Loss=loss.item(), Triplets=miner.num_triplets)

#     # BUG FIX: Save the model into the dedicated MODEL_DIR!
#     save_path = os.path.join(MODEL_DIR, f"audio_resnet_epoch_{epoch+1}.pth")
#     torch.save(train_model.state_dict(), save_path)

In [18]:
model = AudioResNet(embedding_size=128).to(device)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR,"audio_resnet_epoch_20.pth")))

/tmp/ipykernel_10215/2502603131.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(os.path.join(MODEL_DIR,"audio_resnet_epoch_20.pth")))


<All keys matched successfully>

In [19]:
model.eval()

AudioResNet(
  (backbone): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, trac

In [20]:
database_dataset = SimpleAudioDataset(
    data_paths=filenames,
    labels = labels,
    type_of_dataset='original',
    noise_paths = noise_paths
    )

database_dataloader = DataLoader(database_dataset, batch_size=64, shuffle=False)

print("Total filenames:", len(filenames))
print("Total labels:", len(labels))


Total filenames: 40368
Total labels: 40368


In [21]:

all_database_embeddings = []
all_database_labels = []

# Turn off gradient math to save massive amounts of RAM
with torch.no_grad():
    for spectrograms, labels_batch in database_dataloader:
        spectrograms = spectrograms.to(device)
        
        # Get the 128-number coordinates
        embeddings = model(spectrograms)
        
        all_database_embeddings.append(embeddings.cpu())
        all_database_labels.extend(labels_batch.tolist())

# Stack the list of batches into one massive matrix
# Shape will be [Total_Clean_Chunks, 128]
database_matrix = torch.cat(all_database_embeddings, dim=0)
print(f"Database built! Stored {database_matrix.shape[0]} unique fingerprints.")

Database built! Stored 40368 unique fingerprints.


In [22]:
def list_all_sorted(noisy_embedding, database_matrix, database_labels):
    """
    Returns a list of songs in the database compared to the noisy query, 
    sorted from most similar (smallest distance) to least similar.
    """
    # 1. Calculate the distance between the query and EVERY item in the database
    # Output shape is typically [1, N] where N is the number of database items
    distances = torch.cdist(noisy_embedding, database_matrix)
    
    # Flatten the distances to a 1D tensor for easier sorting
    distances_flat = distances.flatten()
    
    # 2. Get the indices that sort the distances in ascending order (smallest first)
    sorted_indices = torch.argsort(distances_flat).tolist()
    
    # 3. Build the sorted list of results
    sorted_results = []
    for idx in sorted_indices:
        predicted_int_label = database_labels[idx]
        predicted_song_name = predicted_int_label
        
        # Grab the actual distance value for this specific match
        distance_val = distances_flat[idx].item()
        
        # Append as a tuple: (Song Name, Distance)
        sorted_results.append((predicted_song_name, distance_val))
    
    return sorted_results

In [23]:
class Evaluator:
    def __init__(self,database_matrix):
        """
        Initializes the evaluator with the dataset.
        """
        self.database_matrix = database_matrix

    def evaluate(self,model,noisy_dataset, test_chunks = None, noise_seed = None):

        

        self.noisy_dataset = noisy_dataset 
        self.model = model

        noisy_dataloader = DataLoader(self.noisy_dataset, batch_size=1, shuffle=True)

        print("\n--- Starting Evaluation Test ---")
        correct_predictions = 0
        total_tests = len(self.noisy_dataset)

        prediction_matrix = []
        real_listed = []

        with torch.no_grad():
            for i, (noisy_spectrogram, true_label) in enumerate(tqdm(noisy_dataloader, desc="🎵 Matching Noisy Queries")):
                noisy_spectrogram = noisy_spectrogram.to(device)
                
                # 1. Get the coordinate for the noisy audio
                noisy_embedding = model(noisy_spectrogram)
                
                # 2. Search the database for the closest match
                predicted_songs_listed = list_all_sorted(
                    noisy_embedding.cpu(), 
                    self.database_matrix, 
                    all_database_labels
                )
                prediction_matrix.append([predicted_song[0] for predicted_song in predicted_songs_listed])
                predicted_song = predicted_songs_listed[0][0]
                # 3. Check if it was right!
                true_song = true_label.item()
                real_listed.append(true_song)
            
            self.ground_truths = real_listed
            self.predictions = prediction_matrix
            self.total_queries = len(self.ground_truths)

    def calculate_top_k(self, k=3):
        """
        Calculates the Top-K Accuracy.
        """
        top_k_hits = 0
        for true_song, pred_list in zip(self.ground_truths, self.predictions):
            if true_song in pred_list[:k]:
                top_k_hits += 1
                
        return top_k_hits / self.total_queries

    def calculate_mrr_at_k(self, k=None):
        """
        Calculates the Mean Reciprocal Rank truncated at K (MRR@K).
        If k is None, evaluates the entire list.
        """
        sum_reciprocal_rank = 0.0
        for true_song, pred_list in zip(self.ground_truths, self.predictions):
            # Truncate the list to top K
            search_list = pred_list[:k] if k is not None else pred_list
            
            if true_song in search_list:
                rank = search_list.index(true_song) + 1  # 1-indexed rank
                sum_reciprocal_rank += 1.0 / rank
                
        return sum_reciprocal_rank / self.total_queries

    def calculate_rank_distribution(self):
        """
        Calculates the percentage distribution of ranks across all provided predictions.
        """
        rank_counts = {}
        misses = 0
        
        for true_song, pred_list in zip(self.ground_truths, self.predictions):
            if true_song in pred_list:
                rank = pred_list.index(true_song) + 1
                rank_counts[rank] = rank_counts.get(rank, 0) + 1
            else:
                misses += 1
                
        distribution = {}
        for rank in sorted(rank_counts.keys()):
            percentage = (rank_counts[rank] / self.total_queries) * 100
            distribution[f"Rank {rank}"] = f"{percentage:.1f}%"
            
        if misses > 0:
            miss_percentage = (misses / self.total_queries) * 100
            distribution["Not Found"] = f"{miss_percentage:.1f}%"
            
        return distribution

    def generate_report_card(self, k_values=[1, 3, 5]):
        """
        Returns a formatted report card evaluating Top-K and MRR@K as a text string.
        """
        report = []
        report.append(f"Total Queries Evaluated: {self.total_queries}\n")
        
        report.append("--- Performance at K Thresholds ---")
        # Clean table header
        report.append(f"{'Threshold':<12} | {'Top-K Accuracy':<15} | {'MRR@K'}")
        report.append("-" * 55)
        
        for k in k_values:
            acc = self.calculate_top_k(k)
            mrr = self.calculate_mrr_at_k(k)
            report.append(f"K = {k:<8} | {acc:>6.2%} ({acc:.3f})   | {mrr:.3f}")
            
        report.append("\n--- Overall Rank Distribution ---")
        distribution = self.calculate_rank_distribution()
        for rank, pct in distribution.items():
            report.append(f"{rank:<12}: {pct}")
            
        # Join all lines into a single master string separated by newlines
        return "\n".join(report)

In [28]:
noisy_dataset = SimpleAudioDataset(
    data_paths=test_filenames,
    labels=test_labels,
    noise_paths=noise_paths,
    type_of_dataset='test'
    )
noisy_dataloader = DataLoader(noisy_dataset, batch_size=1, shuffle=True)

# Initialize the evaluator 
evaluator = Evaluator(database_matrix)
evaluator.evaluate(model, noisy_dataset)
print(evaluator.generate_report_card(k_values=[1, 3, 5]))


--- Starting Evaluation Test ---


🎵 Matching Noisy Queries: 100%|██████████| 448/448 [00:36<00:00, 12.12it/s]

Total Queries Evaluated: 448

--- Performance at K Thresholds ---
Threshold    | Top-K Accuracy  | MRR@K
-------------------------------------------------------
K = 1        | 87.72% (0.877)   | 0.877
K = 3        | 92.63% (0.926)   | 0.899
K = 5        | 94.42% (0.944)   | 0.903

--- Overall Rank Distribution ---
Rank 1      : 87.7%
Rank 2      : 3.3%
Rank 3      : 1.6%
Rank 4      : 0.2%
Rank 5      : 1.6%
Rank 6      : 0.4%
Rank 7      : 0.9%
Rank 9      : 0.2%
Rank 10     : 0.7%
Rank 13     : 0.2%
Rank 23     : 0.2%
Rank 25     : 0.2%
Rank 26     : 0.2%
Rank 28     : 0.4%
Rank 43     : 0.2%
Rank 60     : 0.2%
Rank 74     : 0.2%
Rank 134    : 0.2%
Rank 178    : 0.2%
Rank 261    : 0.2%
Rank 312    : 0.2%
Rank 967    : 0.2%
Rank 998    : 0.2%


In [ ]:

# def shazam_this(audio_file_path, model, database_matrix, database_labels, device):
#     """
#     Takes a raw audio file, processes it, and finds the closest song match.
#     """
#     print(f"Listening to: {audio_file_path}...")
    
#     # 1. Load and force to Mono
#     waveform, sample_rate = torchaudio.load(audio_file_path)
#     if waveform.shape[0] > 1:
#         waveform = torch.mean(waveform, dim=0, keepdim=True)
        
#     # 2. Resample if the user's phone didn't record at 16kHz
#     if sample_rate != 16000:
#         resampler = T.Resample(orig_freq=sample_rate, new_freq=16000)
#         waveform = resampler(waveform)

#     # 3. Convert to Mel Spectrogram
#     mel_transform = T.MelSpectrogram(sample_rate=16000, n_fft=1024, hop_length=512, n_mels=64)
#     amp_to_db = T.AmplitudeToDB()
#     db_mel_spec = amp_to_db(mel_transform(waveform))

#     # 4. Crop or Pad to exactly 157 frames (5 seconds)
#     target_frames = 157
#     current_frames = db_mel_spec.shape[2]
    
#     if current_frames > target_frames:
#         db_mel_spec = db_mel_spec[:, :, :target_frames]
#     elif current_frames < target_frames:
#         padding_amount = target_frames - current_frames
#         db_mel_spec = F.pad(db_mel_spec, (0, padding_amount))

#     # 5. Add a Batch dimension (Shape becomes [1, 1, 64, 157]) and move to GPU
#     db_mel_spec = db_mel_spec.unsqueeze(0).to(device)

#     # 6. Extract the 128-number coordinate and search!
#     with torch.no_grad():
#         embedding = model(db_mel_spec)
#         distances = torch.cdist(embedding.cpu(), database_matrix)
#         closest_index = torch.argmin(distances).item()
        
#     # 7. Decode the result
#     predicted_song = database_labels[closest_index]
#     print(f"🎵 MATCH FOUND: {predicted_song}")
    
#     return predicted_song

In [ ]:
# shazam_this('pevanje.wav',model,database_matrix)